# 09_binding_pocket_analysis

Notebook UI for UPO homolog pocket analysis using an LLM with binding, alignment, and optional reaction inputs.

## Python Path Setup
Ensure project-root imports work whether Jupyter starts from repo root or `notebooks/`.

In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
repo_root = cwd.parent if cwd.name == "notebooks" else cwd
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
src_root = repo_root / "src"
if src_root.exists() and str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

## Imports
Load helper functions for table loading, LLM analysis, output export, and thread persistence.

In [2]:
import importlib
import agentic_protein_design.steps.binding_pocket as bp
bp = importlib.reload(bp)
from project_config.local_api_keys import OPENAI_API_KEY
from agentic_protein_design.core.thread_context import build_thread_context_text
from agentic_protein_design.core import apply_notebook_markdown_style, resolve_input_path

analyze_pocket_profiles = bp.analyze_pocket_profiles
default_user_inputs = bp.default_user_inputs
build_prompt_with_context = bp.build_prompt_with_context
generate_llm_pocket_analysis = bp.generate_llm_pocket_analysis
generate_llm_mutation_design_proposal = bp.generate_llm_mutation_design_proposal
run_llm_pocket_analysis_stages = bp.run_llm_pocket_analysis_stages
prompt_3 = bp.prompt_3
init_thread = bp.init_thread
load_input_tables = bp.load_input_tables
persist_thread_update = bp.persist_thread_update
save_llm_analysis = bp.save_llm_analysis
save_mutation_design_proposal = bp.save_mutation_design_proposal
save_binding_outputs = bp.save_binding_outputs
setup_data_root = bp.setup_data_root
get_step_processed_dir = bp.get_step_processed_dir

apply_notebook_markdown_style(font_size_px=14, line_height=1.4)


## API Key Setup
Load the OpenAI key from `project_config/local_api_keys.py` into environment variables for LLM calls.

In [3]:
if OPENAI_API_KEY and OPENAI_API_KEY != "REPLACE_WITH_YOUR_OPENAI_API_KEY":
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

"OPENAI_API_KEY" in os.environ

True

## User Inputs
Edit all run parameters here (single place): dataset root, thread selection, analysis options, model, and input paths.

In [4]:
root_key = "examples"
existing_thread_key = "binding_pocket_llm_analysis_59353c876ab140688b1c239a15aac24e"  # None

user_inputs = {
    "selected_positions": None, # [100, 103, 104, 107, 141, 222],
    "pairwise_comparisons":  [("CviUPO", "ET096")], # None
    "focus_question": (
        "Identify per-protein structural interpretations and cross-homolog patterns "
        "that could explain activity/property differences."
    ),
    "design_requirements": (
        "Backbone: ET096. Goal: improve peroxygenative mono-oxidation selectivity on S82 "
        "while retaining useful activity and limiting over-oxidation to Di-Ox. "
        "Prioritize conservative, mechanistically justified mutations and a first-round panel <= 12 variants."
    ),
    "literature_context_thread_key": "literature_review_d762a72ec7f04bec9b66ccd3aac21b91",  # Optional: literature-review thread key
    "reaction_data_description": (
        "- Veratryl alcohol: peroxygenative\n"
        "- Naphthalene: peroxygenative\n"
        "- NBD: peroxygenative\n"
        "- ABTS: peroxidative\n"
        "- S82: mixed; Mono-Ox ~ peroxygenation-biased, Di-Ox ~ peroxidation-biased\n"
        "Use ratios (e.g. Mono-Ox : Di-Ox) to infer peroxygenation vs peroxidation balance."
    ),
    "use_reaction_data": True,
    "llm_model": "gpt-5.2",
    "llm_temperature": 0.2,
    "llm_max_rows_per_table": 300,
}

input_paths = {
    # Paths are relative to the data root from project_config.variables.address_dict[root_key].
    "binding_csv": "pdb/bindingpocket_analysis.csv",
    "alignment_csv": "pdb/reps_ali_withDist_FILT.csv",
    "reaction_data_csv": "pdb/substrate_reaction_data.csv",
}

# Optional: reset analysis options from helper defaults
# user_inputs = default_user_inputs()

## Setup Runtime Context
Initialize data directories and active chat thread from the values above.

In [5]:
data_root, resolved_dirs = setup_data_root(root_key)
step_processed_dir = get_step_processed_dir(resolved_dirs)
thread, threads_preview = init_thread(root_key, existing_thread_key)
thread_id = thread["thread_id"]
data_root, step_processed_dir, thread_id


(PosixPath('/Users/charmainechia/Documents/projects/agentic-protein-design/examples'),
 PosixPath('/Users/charmainechia/Documents/projects/agentic-protein-design/examples/processed/09_binding_pocket_analysis'),
 '59353c876ab140688b1c239a15aac24e')

## Load Input Tables
Load descriptor and alignment tables, and optional reaction data, from `input_paths`.

In [6]:
binding_csv = resolve_input_path(data_root, input_paths["binding_csv"])
alignment_csv = resolve_input_path(data_root, input_paths["alignment_csv"])
reaction_data_csv = None
if user_inputs.get("use_reaction_data", False) and input_paths.get("reaction_data_csv", "").strip():
    reaction_data_csv = resolve_input_path(data_root, input_paths["reaction_data_csv"])

pocket, ali, reaction_df = load_input_tables(binding_csv, alignment_csv, reaction_data_csv)
binding_csv, alignment_csv, reaction_data_csv, pocket.head(3), (None if reaction_df is None else reaction_df.head(3))


(PosixPath('/Users/charmainechia/Documents/projects/agentic-protein-design/examples/pdb/bindingpocket_analysis.csv'),
 PosixPath('/Users/charmainechia/Documents/projects/agentic-protein-design/examples/pdb/reps_ali_withDist_FILT.csv'),
 PosixPath('/Users/charmainechia/Documents/projects/agentic-protein-design/examples/pdb/substrate_reaction_data.csv'),
    Unnamed: 0                    struct_name                  struct_name.1  \
 0           0                ET096_S82_glide                ET096_S82_glide   
 1           1               CviUPO_S82_glide               CviUPO_S82_glide   
 2           2  CviUPO-F88L+T158A_S82_chai1_0  CviUPO-F88L+T158A_S82_chai1_0   
 
                    struct_name.2  num_pocket_res_ali  num_pocket_res<6  \
 0                ET096_S82_glide                  38                12   
 1               CviUPO_S82_glide                  39                13   
 2  CviUPO-F88L+T158A_S82_chai1_0                  33                10   
 
    reactive_center_d

## Structured Exports
Generate heuristic comparative tables and export CSVs to `processed/`.

In [7]:
selected_positions = user_inputs["selected_positions"]
interp_df, pattern_summary = analyze_pocket_profiles(pocket, ali, selected_positions)
out_interp, out_patterns = save_binding_outputs(interp_df, pattern_summary, step_processed_dir)


## LLM Pocket Analysis
Query the LLM client with the full prompt and input tables, then save markdown output.

Prerequisite: set `OPENAI_API_KEY` in `project_config/local_api_keys.py`.

In [8]:
# Run two-stage LLM analysis
stage_outputs = run_llm_pocket_analysis_stages(pocket, ali, reaction_df, user_inputs)
prompt_2_output = stage_outputs["prompt_2_output"]
llm_analysis = stage_outputs["combined_analysis"]

out_llm = save_llm_analysis(llm_analysis, step_processed_dir)

# Prompt 3 defaults (overwritten in the next cell)
mutation_design_text = ""
out_mutation_design = None
literature_context_thread_key = None

print(out_llm)


### Binding Pocket Analysis - Stage 1

<details><summary>Prompt</summary>

```text

Analyse the uploaded inputs for a set of proteins to interpret how binding-pocket structure relates to catalytic activity and selectivity. 
Consider how both the proximal (<6 Å from docked ligand) and distal (up to ~11 Å from binding pocket centroid) residues affect the binding pocket environment.

INPUTS
- binding_pocket_table: extracted binding-pocket properties (per protein), calculated separately over proximal and distal residue sets where available.
- pocket_alignment_table: filtered residue alignment of pocket-proximal positions.
- reaction_data (optional): enzyme activity data on substrates.

OBJECTIVE
For each protein, integrate structural descriptors with (optional) reaction data to infer mechanistic behavior and classify pocket phenotypes.

TASKS

1) For each protein:
   - Generate a punchy tagline.
   - Provide a concise 5-6 bullet summary addressing:
        (i) proximal electrostatics  
        (ii) proximal sterics  
        (iii) distal electrostatics  
        (iv) distal sterics / outer pocket size  
        (v) overall synthesis of pocket phenotype, integrating structural properties with catalytic implications:
            - Interpret how geometry and chemistry influence productive (peroxygenative) vs competing (peroxidative) pathways.
            - If reaction_data is provided, use it to support structure–function relationships.

   Use the following column groups:

   PROXIMAL ELECTROSTATICS
   - charged_fraction (proximal), polar_fraction (proximal)
   - kd_weighted (proximal), hw_weighted (proximal)
   - median_dist_res_to_ligand_reactive_center

   PROXIMAL STERICS
   - mean_volume (proximal), weighted_mean_volume (proximal)
   - volume_variance (proximal)
   - small_residue_frac (proximal), bulky_residue_frac (proximal)
   - median_min_dist_res_to_ligand
   - reactive_center_distance
   - num_pocket_res_lt6

   DISTAL ELECTROSTATICS
   - charged_fraction (distal), polar_fraction (distal)
   - kd_weighted (distal), hw_weighted (distal)

   DISTAL STERICS / OUTER POCKET SIZE
   - mean_dist_to_centroid
   - mean_min_dist_to_centroid
   - mean_dist_backbone_to_centroid
   - mean_volume (distal)
   - volume_variance (distal)
   - small_residue_frac (distal), bulky_residue_frac (distal)
   - num_pocket_res_ali

   If proximal/distal suffixes are not explicitly present, infer proximal/distal groupings from context and state your assumption briefly.

2) Comparative analysis requirements (do BOTH):
   A) Intra-protein variant analysis (MANDATORY when variants are present):
   - Detect proteins that share the same base protein identity but differ by variant/mutation labels.
   - For each such protein family, explicitly compare each variant against its WT/reference form (if WT/reference is present).
   - If WT is not explicitly labeled, infer the closest reference sequence in that family and state the assumption.
   - For each variant-vs-reference comparison, report which structural dimensions changed:
        (i) proximal electrostatics
        (ii) proximal sterics
        (iii) distal electrostatics
        (iv) distal sterics / outer pocket size
   - Provide a mechanistic rationale linking those differences to functional shifts.

   B) User-requested pairwise comparisons:
   - Pairwise comparisons requested: CviUPO vs ET096
   - Perform each requested pairwise comparison in addition to section A.
   - Explicitly contrast which structural dimensions changed (prox electrostatics, prox sterics, distal electrostatics, distal sterics).
   - Provide a mechanistic rationale for functional shifts.

3) Distill cross-protein trends or clusters (“pocket phenotypes”):
   - Identify recurring structural archetypes (e.g., tight/polar pose-locking vs open/hydrophobic permissive).
   - Link clusters to turnover vs selectivity trade-offs.

OUTPUT STYLE
- Clear, human-interpretable, mechanistically grounded.
- Emphasize intuition over raw numbers.
- Keep summaries compact and comparative.


REACTION CONTEXT (OPTIONAL)
- Veratryl alcohol: peroxygenative
- Naphthalene: peroxygenative
- NBD: peroxygenative
- ABTS: peroxidative
- S82: mixed; Mono-Ox ~ peroxygenation-biased, Di-Ox ~ peroxidation-biased
Use ratios (e.g. Mono-Ox : Di-Ox) to infer peroxygenation vs peroxidation balance.

REACTION_DATA_STATUS: provided. Rows=5, Columns=['UPO Homologs', 'Veratryl alcohol (peroxygenation)', 'Naphthalene (peroxygenation)', 'NBD (peroxygenation)', 'ABTS (peroxidation)', 'S82 Mono-Ox %yield', 'S82 Di-Ox % Yield', 'S82 Total % Yield', 'S82 Mono:Di Ratio']
```
</details>

#### Response

(Stage 1 output included in compact combined view below.)

### Binding Pocket Analysis - Stage 2

...

#### Response

(Stage 2 output included in compact combined view below.)

### Combined Pocket Analysis

## Stage 1: Global Pocket Phenotypes

## Per-protein integrated interpretations (structure → mechanism → selectivity)

### ET096_S82_glide — **“Small, dry inner pocket; chemistry happens at arm’s length.”**
- **Proximal electrostatics:** Very low polarity/charge (charged_fraction **0.00**, polar_fraction **0.18**); high hydropathy (hw_weighted **−0.94**) and high KD (kd_weighted **2.18**) → a **hydrophobic, poorly H-bonding** near-field. Median residue→reactive-center distance **7.92 Å** suggests few residues directly “coach” the reactive geometry.
- **Proximal sterics:** Many small residues (small_residue_frac **0.64**) with modest mean volume (**88 Å³**) and 12 residues <6 Å; ligand closest approach median **4.33 Å** but reactive center sits relatively far (reactive_center_distance **7.10 Å**) → **roomy microcavity but weak pose-locking at the reactive atom**.
- **Distal electrostatics:** Slightly more polar/charged than proximal (charged **0.13**, polar **0.37**) but still overall hydrophobic (hw_weighted **−0.46**) → distal shell not strongly “electrostatic-funneling”.
- **Distal sterics / outer size:** Outer pocket is relatively extended (mean_dist_to_centroid distal **10.25 Å**; mean_min_dist_to_centroid **8.57 Å**) with moderate distal volumes (~**100 Å³**) → **open outer vestibule**.
- **Phenotype synthesis (with reaction data):** ET096 shows **high S82 di-oxidation** (Mono:Di **0.3**) and modest ABTS (**0.146**). The **hydrophobic, small-residue proximal set + long reactive-center distance** is consistent with **substrate mobility and repeated oxidation** (di-ox) rather than a single, well-registered peroxygenative event. The open distal region likely supports **re-binding/reshuffling**, enabling over-oxidation.

---

### CviUPO_S82_glide — **“Polar-and-bulky clamp near the ligand; tuned for single-hit outcomes.”**
- **Proximal electrostatics:** More polar/charged than ET096 (charged **0.077**, polar **0.385**); still hydrophobic overall (hw_weighted **−0.83**) but with **more H-bonding capacity** near-field. Median residue→reactive-center distance **6.68 Å** (closer than ET096) → better geometric “guidance”.
- **Proximal sterics:** Strongly bulky proximal composition (bulky_residue_frac **0.615**) with high mean volume (**111 Å³**); ligand closest approach median **3.58 Å** and 13 residues <6 Å → **tight, shape-complementary inner pocket** that can enforce a preferred pose.
- **Distal electrostatics:** Distal shell is quite polar (polar **0.513**) with modest charge (0.128) and relatively low KD (kd_weighted **0.387**) → **more hydrophilic outer environment** than ET096, potentially affecting access/solvent organization.
- **Distal sterics / outer size:** Distal centroid distances slightly smaller than ET096 (mean_dist_to_centroid **9.91 Å**, mean_min **8.08 Å**) → **somewhat more compact vestibule**.
- **Phenotype synthesis (with reaction data):** CviUPO has **very high ABTS** (**3.939**) yet S82 is **mono-oxidation biased** (Mono:Di **1.7**). A plausible reconciliation is: the **bulky/polar proximal clamp** favors **a single productive orientation** for S82 (mono), while the **polar/charged distal shell** may stabilize **electron-transfer competent states / solvent networks** that enhance **peroxidative turnover** (ABTS). Net: **selective mono-ox on S82 but strong peroxidase-like behavior on ABTS**.

---

### CviUPO-F88L+T158A_S82_chai1_0 — **“Same scaffold, slightly ‘de-bulky’ and closer to the reactive center.”**
*(Variant family analysis vs CviUPO reference is detailed below; here is the per-protein readout.)*
- **Proximal electrostatics:** Similar charge (charged **0.077**) but **lower polarity** than CviUPO (polar **0.308** vs 0.385); kd_weighted **1.40** (higher than CviUPO’s 1.06) → **more hydrophobic/less H-bonding** proximal field.
- **Proximal sterics:** Mean volume slightly down (**108 Å³**) and fewer residues <6 Å (**10** vs 13). Reactive center is **closer** (reactive_center_distance **5.83 Å**) → fewer contacts, but potentially **more direct access** to the oxidizing center.
- **Distal electrostatics:** Distal remains polar (polar **0.485**) with kd_weighted **0.73** → still relatively hydrophilic outer shell.
- **Distal sterics / outer size:** Distal centroid distances slightly smaller (mean_dist_to_centroid **9.63 Å**) and fewer aligned pocket residues (num_pocket_res_ali **33**) → could reflect **a somewhat simplified/less extensive pocket definition** in this model.
- **Phenotype synthesis:** Without reaction data for this variant, the structural shift suggests a pocket that is **less pose-locking** (fewer <6 Å contacts) but places the reactive center **closer**, which can increase raw reactivity yet risk **less controlled selectivity** depending on substrate.

---

### DcaUPO_S82_glide — **“Reactive-center proximity with a charged outer ring: high activity, mixed pathway pressure.”**
- **Proximal electrostatics:** Low proximal polarity (polar **0.154**) with some charge (0.077); very hydrophobic proximal field (hw_weighted **−1.04**) and high KD (kd_weighted **1.79**) → **nonpolar inner cavity**.
- **Proximal sterics:** Many residues <6 Å (**15**, highest here) and bulky proximal fraction **0.615** with high variance (**1183**) → **tight but heterogeneous** inner pocket; reactive center is close (reactive_center_distance **4.93 Å**) → supports **productive HAT/O-transfer geometry**.
- **Distal electrostatics:** Distal is the **most charged** set (charged **0.20**) with moderate polarity (0.40) → a **charged outer shell** that can influence peroxide/water organization and substrate ingress.
- **Distal sterics / outer size:** Distal centroid distances similar to others (mean_dist_to_centroid **10.00 Å**) with moderate volumes (~**104 Å³**).
- **Phenotype synthesis (with reaction data):** DcaUPO is **high on peroxygenative probes** (Veratryl alcohol **1.558**, NBD **1.242**) but also **high ABTS** (**2.7**); S82 is **mono-biased** (Mono:Di **1.6**). The **short reactive-center distance + many close contacts** fits strong peroxygenation. The **charged distal shell** plausibly promotes **peroxidative competence** (ABTS) by stabilizing ET/solvent networks—yielding a **high-activity, less pathway-exclusive** phenotype.

---

### TE314_S82_chai1_0 — **“Balanced pocket: neither clamp nor cavern, tends toward over-oxidation.”**
- **Proximal electrostatics:** No proximal charge (0.00) and moderate polarity (0.308); kd_weighted **1.82**, hw_weighted **−0.89** → **hydrophobic but not extremely**.
- **Proximal sterics:** Mean volume lower (**98.5 Å³**) with moderate bulky fraction (0.308) and 12 residues <6 Å; reactive center very close (**4.08 Å**) but median residue→reactive-center distance is high (**8.21 Å**) → suggests **a close approach exists but not broadly supported by many residues** (less “caging”).
- **Distal electrostatics:** Distal is fairly hydrophobic (hw_weighted **−0.63**) with kd_weighted **1.43** → **less polar outer shell** than CviUPO/DcaUPO.
- **Distal sterics / outer size:** Distal centroid distances are on the smaller side (mean_dist_to_centroid **9.73 Å**) with relatively low proximal variance (**559**) → **more uniform pocket**.
- **Phenotype synthesis (with reaction data):** S82 Mono:Di **0.7** (di-ox favored). A **less polar, less bulky proximal environment** plus a **not-strongly-structured distal shell** is consistent with **substrate reorientation/rebinding**, enabling sequential oxidation.

---

### OA167_S82_swissdock_0 — **“Bulky hydrophobic inner wall with a permissive vestibule: high total turnover, modest control.”**
- **Proximal electrostatics:** No proximal charge (0.00) but moderate polarity (0.308); very hydrophobic proximal field (hw_weighted **−1.36**, most negative here) with kd_weighted **1.41** → **strongly nonpolar inner pocket**.
- **Proximal sterics:** Bulky proximal fraction **0.538** with high mean volume (**108.5 Å³**) and only 10 residues <6 Å → **hydrophobic packing** but fewer close “steering” contacts; reactive center close (**4.55 Å**).
- **Distal electrostatics:** Distal moderately polar (0.40) and mildly hydrophobic (hw_weighted **−0.67**) → outer shell not strongly charged/polar.
- **Distal sterics / outer size:** Distal centroid distances ~**9.74 Å** (moderate) with typical distal volume (~**101 Å³**) → **reasonably open outer pocket**.
- **Phenotype synthesis (with reaction data):** Highest S82 total yield (**46.8%**) but Mono:Di **0.6** (di-ox favored). This matches a **hydrophobic, permissive pocket** that supports binding/turnover but allows **multiple productive poses over time**, increasing total conversion while sacrificing mono-selectivity.

---

## 2A) Intra-protein variant analysis (families with variants)

### Family: **CviUPO** (reference: **CviUPO_S82_glide**; variant: **CviUPO-F88L+T158A_S82_chai1_0**)  
Assumption: CviUPO_S82_glide is the closest “WT/reference” because it is the unmutated label and shares the same base name.

**Variant vs reference — what changed?**
- **(i) Proximal electrostatics:** **Less polar / more hydrophobic** in variant (polar_fraction **0.385 → 0.308**; kd_weighted **1.06 → 1.40**). Likely reduces H-bond anchoring and increases pose degeneracy.
- **(ii) Proximal sterics:** Slightly **less crowded** (num_pocket_res<6 **13 → 10**) and slightly smaller mean volume (**111 → 108 Å³**), but **reactive center gets closer** (**7.72 → 5.83 Å**). Mechanistically: fewer near contacts can reduce “clamping”, while shorter reactive distance can increase intrinsic oxidation probability once bound.
- **(iii) Distal electrostatics:** Distal becomes **less hydrophilic** (polar **0.513 → 0.485**; kd_weighted **0.387 → 0.73**). This could weaken distal solvent structuring that supports peroxidative ET networks (ABTS-like behavior), potentially shifting balance toward peroxygenation *if* proximal geometry remains productive.
- **(iv) Distal sterics / outer size:** Slightly **more compact** (mean_dist_to_centroid **9.91 → 9.63 Å**) and fewer aligned residues (**39 → 33**), consistent with a subtly altered vestibule definition/shape in the model.

**Mechanistic expectation:** F88L+T158A trends toward a **more hydrophobic, less H-bond-directed pocket** with **more direct access** to the reactive center but **less pose-locking**. For substrates where regio-/stereocontrol depends on tight anchoring, expect **reduced selectivity**; for substrates limited by approach distance, expect **maintained or increased turnover**.

---

## 2B) Requested pairwise comparison: **CviUPO vs ET096**

### CviUPO_S82_glide **vs** ET096_S82_glide
- **Proximal electrostatics:** CviUPO is **much more polar/charged** near the ligand (polar **0.385 vs 0.182**; charged **0.077 vs 0.00**) and has lower KD (kd_weighted **1.06 vs 2.18**) → **better capacity to orient/polarize substrate** and stabilize specific binding modes.
- **Proximal sterics:** CviUPO is **bulkier and tighter** (bulky **0.615 vs 0.273**; mean_volume **111 vs 88 Å³**; median_min_dist **3.58 vs 4.33 Å**) → stronger **shape complementarity/pose restriction**. ET096 has many small residues (small **0.64**) → more “slippery” cavity.
- **Distal electrostatics:** CviUPO distal shell is **more polar** (polar **0.513 vs 0.368**) and less hydrophobic (hw_weighted **−0.48 vs −0.46**, similar) but notably lower KD (kd_weighted **0.387 vs 0.967**) → **more hydrophilic vestibule** in CviUPO.
- **Distal sterics / outer size:** ET096 is **more extended/open** distally (mean_dist_to_centroid **10.25 vs 9.91 Å**; mean_min_dist **8.57 vs 8.08 Å**) → easier access and potentially more re-binding/reorientation.

**Mechanistic rationale tied to reaction data:**
- ET096’s **open, hydrophobic, small-residue proximal pocket** aligns with **di-oxidation dominance on S82** (Mono:Di **0.3**)—substrate can reorient and get hit multiple times.
- CviUPO’s **polar + bulky proximal clamp** aligns with **mono-oxidation bias** (Mono:Di **1.7**) by enforcing a preferred pose and limiting over-oxidation.
- CviUPO’s **high ABTS** (3.939 vs 0.146) is consistent with its **more polar/charged pocket environment** (proximal and distal), which can support **peroxidative electron-transfer chemistry** and/or stabilize reactive intermediates/solvent networks relative to ET096’s “dry” pocket.

---

## 3) Cross-protein “pocket phenotypes” (recurring archetypes → turnover/selectivity trade-offs)

### Phenotype 1: **Hydrophobic–permissive cavities (mobility → over-oxidation)**
- **Members:** ET096, TE314, OA167 (strongest in ET096/OA167 by proximal hydropathy and/or low proximal charge).
- **Signature:** Low proximal charge, relatively hydrophobic hw_weighted, fewer polar anchors; often more open distal geometry.
- **Functional tendency:** Higher likelihood of **multiple binding poses and sequential oxidation** → **lower mono-selectivity (lower Mono:Di)** but can give **high total conversion** (OA167).

### Phenotype 2: **Bulky/polar pose-locking inner pockets (control → mono-selectivity)**
- **Members:** CviUPO, DcaUPO (both have high bulky proximal fraction; CviUPO has highest proximal polarity).
- **Signature:** High bulky_residue_frac proximal, closer median_min_dist to ligand, and increased proximal polarity/charge.
- **Functional tendency:** Better **regio-/chemoselective “single-hit” outcomes** on S82 (higher Mono:Di), but can still support peroxidation if distal shell is polar/charged (CviUPO, DcaUPO high ABTS).

### Phenotype 3: **Charged distal shell “peroxidation enabler”**
- **Members:** DcaUPO (charged distal **0.20**), CviUPO (high distal polarity).
- **Signature:** Elevated distal charge/polarity with relatively hydrophilic KD.
- **Functional tendency:** Promotes **peroxidative competence** (ABTS) by stabilizing ET/solvent organization, sometimes at the cost of pathway exclusivity (DcaUPO: high peroxygenation *and* high ABTS).

If you want, I can also (i) map the pocket_alignment_table positions onto these phenotypes (which specific aligned sites are likely “gatekeepers”), and (ii) propose 2–3 mutation ideas per phenotype to push toward mono-selective peroxygenation vs peroxidation suppression.

## Stage 2: Residue-Level Mechanistic Drivers

## 1) Key variable pocket positions → residue-level mechanistic hypotheses
(Positions are given in each protein’s **own numbering** as requested; I refer back to the earlier pocket phenotypes: **ET096/TE314/OA167 = hydrophobic–permissive/over-oxidation**, **CviUPO/DcaUPO = bulky/polar clamp/mono-bias**, **DcaUPO/CviUPO distal polarity/charge = peroxidation enabler**.)

### A. “Clamp vs permissive wall” near the ligand: **CviUPO 88 / ET096 103 / DcaUPO 86 / TE314 108 / OA167 104**
- **Identities**
  - ET096 **I103**
  - CviUPO **F88** → variant **L88** (F88L)
  - DcaUPO **L86**
  - TE314 **I108**
  - OA167 **I104**
- **Substitution class**
  - **Steric/aromaticity shift:** F ↔ (I/L). Phenylalanine is bulkier and π-capable; Leu/Ile are smaller, purely aliphatic.
  - **Polarity:** all hydrophobic; main change is **shape + π interactions**, not charge.
- **Mechanistic consequence**
  - **CviUPO F88** is a classic **pose-locking “clamp” element**: aromatic face can pack against hydrophobic substrate and reduce pose degeneracy → consistent with CviUPO’s **bulky/polar clamp** phenotype and **mono-oxidation bias** (Mono:Di 1.7).
  - **F88L (variant)** removes π-stacking and slightly reduces sidechain volume → **weakens clamping**, increases microcavity “slipperiness,” matching the summary: **fewer <6 Å contacts** and **lower proximal polarity** → predicted **reduced selectivity / more reorientation**, even if reactive-center access improves.
- **Within-family contrast (CviUPO vs F88L+T158A)**
  - **F88→L88** specifically “de-aromatizes” the clamp: expect **less enforced substrate orientation** (more trajectories that still reach the oxidant), aligning with the variant’s “less pose-locking” description.

**Confidence:** High (directly matches “bulky clamp” vs “de-bulky” narrative and is a large physicochemical change at a proximal site).

---

### B. “Electrostatic gate / distal-shell charge injector”: **CviUPO 165 / ET096 178 / DcaUPO 161 / TE314 190 / OA167 181**
- **Identities**
  - ET096 **A178**
  - CviUPO **K165** (also **K165** in variant)
  - DcaUPO **C161**
  - TE314 **V190**
  - OA167 **A181**
- **Substitution class**
  - **Electrostatic:** Lys (**+1**) vs A/V/C (neutral). This is the strongest explicit charge difference in the table.
  - **Steric:** K is also longer/bulkier than A/V/C.
- **Mechanistic consequence**
  - **CviUPO K165** can create a **local positive electrostatic patch** that:
    - stabilizes/organizes **water/peroxide networks** and polar transition states (supporting the earlier “polar/charged environment → ABTS competence”),
    - can **electrostatically steer** polar substrate moieties or constrain approach vectors (a “soft gate”).
  - In **ET096/TE314/OA167** (A/V/A) the same region is **electrostatically silent**, consistent with their more “dry/permissive” phenotypes and greater tendency toward **reorientation → di-oxidation**.
  - **DcaUPO C161** is neutral but polarizable; it won’t replicate the strong distal/proximal electrostatic steering of Lys—consistent with DcaUPO’s distal charge being distributed elsewhere (summary: **charged distal shell** overall), not necessarily at this exact site.
- **Within-family contrast**
  - No change between CviUPO and its variant at 165, so **K165 likely preserves** part of CviUPO’s electrostatic “peroxidation-enabling” character even as F88L/T158A reduce pose-locking/polar anchoring elsewhere.

**Confidence:** High for electrostatics/pathway bias (charged vs neutral at pocket edge is a canonical driver).

---

### C. “Hydrogen-bond anchor vs hydrophobic release” at the T158A mutation site: **CviUPO 158 / ET096 171 / DcaUPO 154 / TE314 183 / OA167 174**
- **Identities**
  - ET096 **A171**
  - CviUPO **T158** → variant **A158** (T158A)
  - DcaUPO **F154**
  - TE314 **V183**
  - OA167 **P174**
- **Substitution class**
  - **Polarity/H-bonding:** Thr (polar, H-bond donor/acceptor) → Ala (nonpolar).
  - **Steric:** small-to-small (minor volume change), but **loss of hydroxyl** is major chemically.
- **Mechanistic consequence**
  - In **CviUPO (T158)**: provides a **specific H-bonding handle** that can “register” substrate orientation and/or stabilize a local water network → consistent with the **polar proximal clamp** phenotype and mono-selectivity.
  - **T158A (variant)** removes that anchor → **reduced H-bond-directed positioning**, increased pose degeneracy and potentially increased radical/oxygen rebound variability. This directly matches the summary: variant becomes **less polar / more hydrophobic** and **less pose-locking**.
  - Cross-protein context: ET096 already has **A171** (no anchor) and shows **di-oxidation dominance**; T158A pushes CviUPO **toward the ET096-like “dry/permissive” behavior**.
- **Within-family contrast**
  - This is the cleanest causal link to the variant’s reported **polarity drop** (0.385 → 0.308): **T158A is a primary driver** of that shift.

**Confidence:** High (directly changes H-bond capacity at a pocket residue and aligns with the observed phenotype shift).

---

### D. “Bulky plug vs small hinge” controlling local crowding: **ET096 80 / CviUPO 64 / DcaUPO 62 / TE314 84 / OA167 80**
- **Identities**
  - ET096 **A80**
  - CviUPO **L64**
  - DcaUPO **F62**
  - TE314 **P84**
  - OA167 **L80**
- **Substitution class**
  - **Steric:** A (small) ↔ L/P (medium) ↔ F (bulky aromatic).
  - **Polarity:** all largely nonpolar (Pro is nonpolar but conformationally special).
- **Mechanistic consequence**
  - **ET096 A80** contributes to the “small-residue proximal set” → **roomier microcavity**, weaker caging → consistent with **substrate mobility and di-oxidation**.
  - **DcaUPO F62** is a **bulky plug** that can tighten the inner pocket and enforce approach geometry (fits DcaUPO’s **many <6 Å contacts** and close reactive center).
  - **TE314 P84** can rigidify a loop/turn and shape the pocket wall; proline often acts as a **conformational gate** (less about volume, more about fixing backbone geometry), potentially explaining TE314’s “close approach exists but not broadly supported” (a localized gate rather than a global clamp).

**Confidence:** Medium-high (strong steric differences; exact effect depends on sidechain orientation/backbone context).

---

### E. “Charge/polarity hotspot” at a near-ligand position: **ET096 77 / CviUPO 60 / DcaUPO 58 / TE314 80 / OA167 76**
- **Identities**
  - ET096 **A77**
  - CviUPO **T60** (variant also T60)
  - DcaUPO **D58**
  - TE314 **T80**
  - OA167 **A76**
- **Substitution class**
  - **Electrostatic:** D (−1) vs A/T (neutral).
  - **Polarity:** T is polar; A is nonpolar.
- **Mechanistic consequence**
  - **DcaUPO D58** introduces a **fixed negative charge** near the pocket that can:
    - stabilize cationic/polar substrate features,
    - tune local protonation/water structure, potentially supporting DcaUPO’s **mixed peroxygenation + peroxidation pressure** (summary: charged distal shell; this is one concrete contributor).
  - ET096/OA167 (A) lack this, consistent with “dry” permissive cavities.
  - CviUPO/TE314 (T) provide **H-bonding without full charge**, consistent with intermediate polarity.

**Confidence:** Medium (clear electrostatic difference, but distances here are somewhat larger in some structures; still within the filtered pocket set).

---

## 2) Ranked residue list (mechanistic drivers vs modulators vs neutral)

### High-confidence mechanistic driver residues
1. **CviUPO K165** (vs ET096 A178 / TE314 V190 / OA167 A181 / DcaUPO C161): **charge-based electrostatic gating/solvent organization** → ties to CviUPO/DcaUPO peroxidation competence vs ET096-like dryness.
2. **CviUPO T158A (variant)** at **158**: **loss of H-bond anchor** → directly explains variant’s **reduced proximal polarity** and predicted **reduced pose-locking/selectivity**.
3. **CviUPO F88L (variant)** at **88**: **aromatic clamp removal** → reduces shape/π complementarity, increasing pose degeneracy (variant shift toward permissive behavior).

### Secondary modulators (steric shaping; context-dependent)
- **ET096 A80 / CviUPO L64 / DcaUPO F62 / TE314 P84 / OA167 L80**: steric “plug/hinge” controlling local crowding and backbone rigidity.
- **DcaUPO D58** (vs A/T): localized **negative electrostatic feature** that can bias binding/solvent structure.
- **ET096 V74 / OA167 T73 / others L**: small polarity tweak (Thr vs hydrophobe) likely modulates hydration/entry but less decisive alone.

### Likely neutral/background (in this dataset; weaker causal leverage)
- Positions dominated by conservative hydrophobes with similar size and no charge change, e.g. **ET096 223 F vs others M/L/V/I** (mostly hydrophobic packing differences), and **ET096 172/173 region (S/A vs S/T/Y/L)** which are relatively distal in the provided distances and may not strongly control the reactive geometry unless they line the access channel in a specific structure.

If you want, I can convert these into **testable mutation hypotheses** (e.g., “make ET096 more mono-selective by introducing a Thr/Lys at the K165-equivalent site + adding an aromatic clamp at the F88-equivalent site”) while keeping each proposal consistent with the phenotypes you summarized.

/Users/charmainechia/Documents/projects/agentic-protein-design/examples/processed/09_binding_pocket_analysis/binding_pocket_llm_analysis.md


## LLM Backbone Engineering Proposal
Use Stage-2 residue-level drivers plus optional literature-thread context to propose mutation designs under user requirements.

In [9]:
design_requirements = str(user_inputs.get("design_requirements", "")).strip()
literature_context_thread_key = str(user_inputs.get("literature_context_thread_key", "")).strip() or None

context_result = build_thread_context_text(
    literature_context_thread_key,
    include_referenced_files=True,
    max_chars_per_file=20000,
    on_missing="warn",
)
literature_context = str(context_result.get("context_text", ""))
literature_context_bundle = context_result.get("context_bundle")

mutation_design_text = generate_llm_mutation_design_proposal(
    prompt_2_output=prompt_2_output,
    design_requirements=design_requirements,
    user_inputs=user_inputs,
    literature_context=literature_context,
)
out_mutation_design = save_mutation_design_proposal(mutation_design_text, step_processed_dir)
out_mutation_design


### Binding Pocket Mutation Design Proposal

<details><summary>Prompt</summary>

```text

You are designing enzyme variants for rational engineering.

You are given:
1) prompt_2_output: residue-level mechanistic analysis of binding-pocket drivers.
2) literature_context (optional): prior external context (for example literature-review thread outputs).
3) design_requirements: user-provided requirements including:
   - target backbone protein to engineer
   - engineering aims (activity/selectivity/stability/pathway bias)
   - constraints (allowed positions, mutation budget, excluded residues/motifs, expression or assay limits)

TASK
Generate a concrete mutation design proposal grounded primarily in prompt_2_output and supported by literature_context when relevant.

OUTPUT FORMAT
1) Design Intent
   - State backbone protein and explicit engineering objective.

2) Proposed Mutations (ranked)
   - Provide 5-10 proposals total.
   - Include both:
     - specific substitutions (e.g., F88L), and
     - optional position-level exploration suggestions (e.g., site-saturation at position 158 with a small focused set).
   - For each proposal provide:
     - rank
     - proposal (mutation or position-set)
     - rationale linked to prompt_2 mechanistic driver(s)
     - expected directional effect on function
     - risk/tradeoff
     - confidence (high/medium/low)

3) Minimal Experimental Plan
   - Suggest a compact first-round panel (6-12 variants max), prioritizing high-information designs.
   - Include a short assay/readout plan aligned with the objective.

4) Rejected Alternatives
   - Briefly list 3-5 plausible but lower-priority options and why they were deprioritized.

RULES
- Do not invent residue numbering outside the provided context.
- Keep causal links explicit from residue-level mechanism -> mutation -> expected phenotype.
- If literature_context conflicts with prompt_2_output, state the conflict and choose a conservative design.
- Prefer practical, testable proposals over speculative broad recommendations.

```
</details>

#### Response

(Full mutation proposal shown below in compact view.)

### Mutation Design Proposal

## 1) Design Intent
- **Backbone protein:** **ET096**
- **Objective:** Shift ET096 from a **hydrophobic/permissive, over-oxidation–prone pocket** toward a more **pose-locking / mono-oxidation–biased** catalyst for **peroxygenative mono-oxidation on S82**, while **retaining useful activity** and **reducing Di-Ox (over-oxidation)**.

Mechanistic basis from prompt_2_output: ET096’s permissive/“dry” behavior is linked to (i) lack of a **proximal H-bond anchor** at **A171** (vs CviUPO T158), (ii) lack of a **distal-shell positive electrostatic gate** at **A178** (vs CviUPO K165), and (iii) a **roomier steric environment** at **A80** (vs bulkier/rigidifying residues in other UPOs). We will introduce **minimal polarity/steric “clamping”** at these ET096 positions.

---

## 2) Proposed Mutations (ranked)

### 1) **A171T**
- **Rationale (driver):** Position **ET096 171 = A** corresponds to the **T158 anchor site** in CviUPO; prompt_2 links Thr here to **H-bond anchoring/pose registration** and mono-bias, while Ala correlates with **pose degeneracy and di-oxidation**.
- **Expected effect:** **Increase mono-oxidation selectivity**, reduce Di-Ox by reducing substrate reorientation/rebinding.
- **Risk/tradeoff:** Could **reduce kcat** if the new H-bond network over-constrains binding or perturbs local water structure.
- **Confidence:** **High**

### 2) **A178K**
- **Rationale (driver):** ET096 has **A178** where CviUPO has **K165**, described as an **electrostatic gate / solvent organizer** supporting a more polar/charged environment and pathway bias away from “dry permissive” behavior.
- **Expected effect:** **Reduce over-oxidation** by (i) increasing organized polarity near pocket edge (less “slippery” trajectories) and (ii) potentially altering peroxide/water organization to favor productive peroxygenation over repeated turnovers on product.
- **Risk/tradeoff:** Lys introduction can **destabilize** (buried charge) or **increase peroxidative side activity** depending on how it couples to electron-transfer/water networks (literature notes ABTS is a sensitive peroxidase reporter; this could move the wrong way).
- **Confidence:** **Medium-high** (strong mechanistic lever, but charge burial risk)

### 3) **A80L**
- **Rationale (modulator):** Prompt_2 flags ET096 **A80** as a “small hinge” contributing to a **roomier microcavity** and mobility/di-oxidation. Moving toward **L (as in CviUPO/OA167)** should partially **tighten** and reduce pose multiplicity without extreme bulk.
- **Expected effect:** **Improved mono-selectivity** (more caging), modest activity impact.
- **Risk/tradeoff:** Could **reduce substrate access** if this region is part of the entry path; may lower total turnover.
- **Confidence:** **Medium**

### 4) **A80P**
- **Rationale (modulator):** TE314 has **P84**, proposed to act as a **conformational gate** (rigidifies local backbone). Proline can reduce “breathing” that enables reorientation/rebinding.
- **Expected effect:** **Lower Di-Ox** by restricting pocket dynamics; may sharpen product profile.
- **Risk/tradeoff:** Proline can be **structurally disruptive** (backbone strain) and harm expression/folding.
- **Confidence:** **Low-medium**

### 5) **A77T**
- **Rationale (modulator):** ET096 **A77** sits at a “charge/polarity hotspot” where CviUPO/TE314 have **T** (H-bonding without full charge). This is a conservative way to **increase proximal polarity** without introducing a formal charge (vs D58 in DcaUPO).
- **Expected effect:** Slight **increase in pose registration** / hydration control → **reduced over-oxidation**.
- **Risk/tradeoff:** Effect may be **small** alone.
- **Confidence:** **Medium**

### 6) **A171T + A178K** (double)
- **Rationale:** Combine the two **highest-confidence mechanistic drivers**: add **H-bond anchor** (171) + **electrostatic gate** (178). This is the most direct “ET096 → more clamp-like” conversion per prompt_2.
- **Expected effect:** Strongest predicted **mono-bias**; may also improve coupling/productive trajectories.
- **Risk/tradeoff:** Higher chance of **activity loss** or **mis-tuned peroxidation** (monitor ABTS/NBD-type split; see plan).
- **Confidence:** **Medium-high**

### 7) **A171T + A80L** (double)
- **Rationale:** Pair **pose anchor** (171T) with **steric tightening** (80L) to reduce both rotational freedom and microcavity “slip.”
- **Expected effect:** **Mono-selectivity up**, Di-Ox down; potentially better than either alone.
- **Risk/tradeoff:** Could **over-restrict** and reduce conversion on S82.
- **Confidence:** **Medium**

### 8) **A178K + A80L** (double)
- **Rationale:** Combine **electrostatic steering/solvent organization** (178K) with **steric caging** (80L).
- **Expected effect:** Reduced Di-Ox; may preserve activity better than adding the 171T anchor (depends on S82’s functional groups).
- **Risk/tradeoff:** Same charge-burial concern as A178K; plus possible access limitation.
- **Confidence:** **Medium**

### 9) **Focused exploration at 171: {A171T, A171S}**
- **Rationale:** Prompt_2’s key is “H-bond anchor vs none.” **Ser** is a more conservative anchor than Thr (less steric), sometimes better if space is tight.
- **Expected effect:** Tune mono-bias vs activity tradeoff.
- **Risk/tradeoff:** Requires 1–2 extra constructs; still small.
- **Confidence:** **Medium**

### 10) **Focused exploration at 178: {A178K, A178R}**
- **Rationale:** If positive charge is beneficial, **Arg** can provide a different geometry/H-bonding pattern than Lys (sometimes less destabilizing depending on burial and H-bond partners).
- **Expected effect:** Similar direction as K; may improve stability or reduce unintended peroxidation.
- **Risk/tradeoff:** Arg can be even harder to bury; could worsen expression.
- **Confidence:** **Low-medium**

---

## 3) Minimal Experimental Plan

### First-round variant panel (≤12; high-information)
Include WT as baseline (not counted as a “variant” if you prefer, but include in assays).

1. **A171T**
2. **A171S** (anchor strength titration)
3. **A178K**
4. **A178R** (charge geometry test)
5. **A80L**
6. **A80P** (dynamic gate test; higher risk but informative)
7. **A77T**
8. **A171T/A178K**
9. **A171T/A80L**
10. **A178K/A80L**
11. **A171T/A77T** (anchor + local polarity)
12. **A171T/A178K/A80L** (triple “clamp package”; only if expression is acceptable—otherwise swap for **A171T/A178K + A77T**)

### Assay/readouts aligned to objective (mono-oxidation on S82; suppress Di-Ox)
- **Primary analytics:** Quantify **Mono-Ox vs Di-Ox** on **S82** by **LC-MS (or GC-MS if derivatized/volatile)** at multiple timepoints (early + near-complete conversion). Report **Mono:Di ratio** and **TTN**.
- **Coupling / side-pathway counterscreen (literature-supported):**
  - Run a **peroxidase reporter** (e.g., **ABTS oxidation**) in parallel to detect variants drifting toward 1e⁻ chemistry (literature_context highlights ABTS as robust peroxidase readout).
  - If available, include a **peroxygenation reporter** (e.g., **NBD**) to ensure you’re not selecting “low ABTS because dead enzyme.”
- **Process control (to avoid confounding):** Use **controlled H₂O₂ delivery** (fed-batch or low steady-state) because overoxidation and inactivation are peroxide-sensitive in UPOs (literature_context). Keep identical peroxide profiles across variants.

Decision rule after round 1: advance variants that **increase Mono:Di** at matched conversion (or matched TTN) and do **not** show a disproportionate increase in ABTS activity relative to S82 peroxygenation.

---

## 4) Rejected Alternatives (deprioritized)
1. **Introduce a negative charge at 77 (A77D/E)**  
   - Prompt_2 notes DcaUPO has a D at the analogous site (D58), but ET096 is in the “dry/permissive” cluster; adding a negative charge is **less conservative** and could unpredictably alter peroxide/water networks or destabilize.
2. **Large aromatic “clamp” insertions at ET096 positions not explicitly mapped**  
   - Prompt_2’s aromatic clamp discussion is for **CviUPO F88** vs aliphatic; ET096’s exact equivalent position is not provided here, so proposing new aromatic clamps would violate the “don’t invent numbering/mapping” constraint.
3. **Multi-site broad hydrophobic repacking (many conservative hydrophobe swaps)**  
   - Prompt_2 labels many hydrophobe-only positions as likely **background/neutral**; these are lower leverage than the clear driver sites (171, 178) and would burn mutation budget.
4. **Aggressive pocket plugging with Phe at 80 (A80F)**  
   - Although DcaUPO has a bulky aromatic at the analogous site (F62), this is a **large steric jump** likely to crush activity on S82; start with **L** (and optionally P) first.

If you can share whether S82 has a polar handle (H-bond acceptor/donor) near the oxidation site, I’d prioritize **A171T vs A171S** differently (Thr is better for stronger registration; Ser is safer if space is tight).

PosixPath('/Users/charmainechia/Documents/projects/agentic-protein-design/examples/processed/09_binding_pocket_analysis/binding_pocket_mutation_design.md')

## Save Thread Update
Run this final cell to append run metadata and prompt context to `chats/<llm_process_tag>_<thread_id>.json`.

In [10]:
persist_thread_update(
    root_key=root_key,
    thread_id=thread_id,
    user_inputs=user_inputs,
    input_paths=input_paths,
    selected_positions=selected_positions,
    reaction_df=reaction_df,
    out_interp=out_interp,
    out_patterns=out_patterns,
    llm_analysis_path=out_llm,
    llm_analysis_text=llm_analysis,
    mutation_design_path=out_mutation_design,
    mutation_design_text=mutation_design_text,
    literature_context_thread_key=literature_context_thread_key,
    llm_model=str(user_inputs.get("llm_model", "")),
)

'2026-02-24T07:54:32.350750+00:00'